# First Agent — ReAct (Reasoning + Action)

Notebook do curso. Implementa loop **Thought → Action → Observation → Answer**.

## Setup (uma vez)

No terminal, dentro da pasta `first-agent`:

```bash
uv python install 3.11        # garante Python 3.11
uv sync                       # cria .venv e instala deps do pyproject.toml
cp .env.example .env          # depois edite .env e coloque sua chave
uv run python -m ipykernel install --user --name first-agent --display-name "Python (first-agent)"
```

Depois abra este notebook e selecione o kernel **Python (first-agent)**.

Para trocar de versão Python no futuro: edite `requires-python` no `pyproject.toml` e rode `uv sync` de novo.

## 1. Importações e configuração

Carrega `.env`, escolhe provider (`openai` ou `gemini`), checa chave.

In [17]:
import os
import re
from dotenv import load_dotenv

load_dotenv()

PROVIDER = os.getenv("LLM_PROVIDER", "gemini").lower()
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

if PROVIDER == "openai":
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY ausente no .env"
    from openai import OpenAI
    _client = OpenAI()
elif PROVIDER == "gemini":
    assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY ausente no .env"
    from google import genai
    from google.genai import types as genai_types
    _gemini = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
else:
    raise ValueError(f"LLM_PROVIDER desconhecido: {PROVIDER}")

print(f"Provider ativo: {PROVIDER}")

Provider ativo: gemini


## 2. Classe Agent

Guarda histórico de mensagens. Método `__call__` registra mensagem e chama LLM. `execute` faz a chamada de fato com `temperature=0` (resultados determinísticos).

In [18]:
class Agent:
    def __init__(self, system: str = ""):
        self.system = system
        self.messages: list[dict] = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message: str) -> str:
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self) -> str:
        if PROVIDER == "openai":
            resp = _client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=0,
                messages=self.messages,
            )
            return resp.choices[0].message.content

        # gemini (google-genai SDK): system_instruction separado, roles user/model
        system_msg = next((m["content"] for m in self.messages if m["role"] == "system"), None)
        contents = []
        for m in self.messages:
            if m["role"] == "system":
                continue
            role = "model" if m["role"] == "assistant" else "user"
            contents.append({"role": role, "parts": [{"text": m["content"]}]})
        resp = _gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=contents,
            config=genai_types.GenerateContentConfig(
                system_instruction=system_msg,
                temperature=0,
            ),
        )
        return resp.text.strip()

## 3. Prompt ReAct

Ensina o agente a alternar **Thought → Action → PAUSE → Observation → Answer**.

In [19]:
prompt = """\
Você opera em um loop de Thought, Action, PAUSE, Observation.
No final do loop você produz um Answer.
Use Thought para descrever seu raciocínio sobre a pergunta.
Use Action para executar uma das ações disponíveis — então retorne PAUSE.
Observation será o resultado da execução dessas ações.

Ações disponíveis:

calculate:
ex: calculate: 4 * 7 / 3
Executa expressão aritmética com eval() do Python. Use ponto para decimais.

preco_prato:
ex: preco_prato: feijoada
Retorna o preço de um prato do cardápio.

Exemplo de sessão:

Question: Quanto custa uma feijoada?
Thought: Preciso buscar o preço da feijoada no cardápio.
Action: preco_prato: feijoada
PAUSE

Você será chamado novamente com:

Observation: O prato feijoada custa R$ 45.

Você então responde:

Answer: A feijoada custa R$ 45.
""".strip()

## 4. Ações (tools)

Funções concretas que o agente pode invocar. Mapeadas em `known_actions`.

In [20]:
def calculate(what: str) -> float:
    return eval(what)

PRECOS = {
    "feijoada": 45,
    "picanha": 80,
    "moqueca": 60,
    "estrogonofe": 35,
    "lasanha": 40,
}

def preco_prato(nome: str) -> str:
    nome = nome.strip().lower()
    if nome in PRECOS:
        return f"O prato {nome} custa R$ {PRECOS[nome]}."
    return f"Prato '{nome}' não encontrado no cardápio."

known_actions = {
    "calculate": calculate,
    "preco_prato": preco_prato,
}

## 5. Fluxo manual (didático)

Roda o ciclo na mão pra ver cada passo: Thought → Action → Observation → Answer.

In [21]:
bot = Agent(prompt)
resposta = bot("Quanto custa uma feijoada?")
print(resposta)

Action: preco_prato: feijoada
PAUSE


In [22]:
# Simula a observação manualmente e devolve ao agente
obs = preco_prato("feijoada")
print("Observation:", obs)
resposta_final = bot(f"Observation: {obs}")
print(resposta_final)

Observation: O prato feijoada custa R$ 45.
Answer: A feijoada custa R$ 45.


## 6. Automação do ciclo — `query()`

Detecta linhas `Action: nome: input`, executa a ação, devolve observação, repete até receber `Answer:`.

In [23]:
action_re = re.compile(r"^Action: (\w+): (.*)$")

def query(question: str, max_turns: int = 5) -> str | None:
    bot = Agent(prompt)
    next_prompt = question
    for turn in range(max_turns):
        result = bot(next_prompt)
        print(f"--- turn {turn+1} ---")
        print(result)
        actions = [action_re.match(line) for line in result.split("\n") if action_re.match(line)]
        if not actions:
            return result  # contém o Answer
        action, action_input = actions[0].groups()
        if action not in known_actions:
            raise Exception(f"Ação desconhecida: {action}: {action_input}")
        print(f" -> executando {action}({action_input!r})")
        observation = known_actions[action](action_input)
        print(f" <- Observation: {observation}")
        next_prompt = f"Observation: {observation}"
    return None

In [24]:
query("Quanto custa uma feijoada?")

--- turn 1 ---
Action: preco_prato: feijoada
PAUSE
 -> executando preco_prato('feijoada')
 <- Observation: O prato feijoada custa R$ 45.
--- turn 2 ---
Answer: A feijoada custa R$ 45.


'Answer: A feijoada custa R$ 45.'

## 7. Pergunta composta

Força o agente a encadear múltiplas ações: dois `preco_prato` + um `calculate`.

In [25]:
query("Qual o valor total de uma feijoada com uma picanha?")

--- turn 1 ---
Action: preco_prato: feijoada
PAUSE
 -> executando preco_prato('feijoada')
 <- Observation: O prato feijoada custa R$ 45.
--- turn 2 ---
Action: preco_prato: picanha
PAUSE
 -> executando preco_prato('picanha')
 <- Observation: O prato picanha custa R$ 80.
--- turn 3 ---
Thought: Agora que tenho os preços da feijoada (R$ 45) e da picanha (R$ 80), preciso somá-los para obter o valor total.
Action: calculate: 45 + 80
PAUSE
 -> executando calculate('45 + 80')
 <- Observation: 125
--- turn 4 ---
Answer: O valor total de uma feijoada com uma picanha é R$ 125.


'Answer: O valor total de uma feijoada com uma picanha é R$ 125.'